## **Case #001: The Phantom Purchase Order**

## 

**Database**: _AdventureWorks2019_

### **Brief Case**

## 

In the Denver distribution center, the finance department at AdventureWorks has detected a strange anomaly — a large purchase order was processed, but no corresponding shipment or inventory deduction can be found. <span style="background-color: rgba(127, 127, 127, 0.1); color: var(--vscode-foreground);">The transaction occurred on&nbsp;</span> **March 25, 2014**<span style="color: var(--vscode-foreground);">. Your task is to trace the transaction, identify any suspicious activity, and confirm if an employee may have faked the purchase.</span>

### **Objectives**

## 

1. Retrieve the purchase order record from the `Purchasing.PurchaseOrderHeader` table for the given date.
    
2. Check the `Inventory` and `PurchaseOrderDetail` tables for inconsistencies in item movement.
    
3. Identify the employee who created or approved the order using `Employee` or `Vendor` tables.
    
4. Review the employee’s department history and/or order history for that day to confirm intent or fraud.

## **SQL Workspace to Test Queries**

In [1]:
USE AdventureWorks2019
GO

-- Step 1: Retrieve the suspicious purchase order(s) on a specific date
SELECT *
FROM Purchasing.PurchaseOrderHeader
WHERE OrderDate = '2014-03-25';

-- Step 2: Check the order details for inconsistencies in item quantity or price
SELECT pph.PurchaseOrderID, pod.ProductID, pod.OrderQty, pod.UnitPrice, pi.Quantity
FROM Purchasing.PurchaseOrderHeader AS pph
INNER JOIN Purchasing.PurchaseOrderDetail AS pod 
    ON pph.PurchaseOrderID = pod.PurchaseOrderID
INNER JOIN Production.ProductInventory AS pi 
    ON pod.ProductID = pi.ProductID
WHERE pph.OrderDate = '2014-03-25';

-- Step 3: Identify the employee who created or approved the order using Vendor table and EmployeeID
SELECT pph.PurchaseOrderID, pph.EmployeeID, v.BusinessEntityID AS VendorID, v.Name AS VendorName, v.CreditRating
FROM Purchasing.PurchaseOrderHeader AS pph
INNER JOIN Purchasing.Vendor AS v 
    ON pph.VendorID = v.BusinessEntityID
WHERE pph.OrderDate = '2014-03-25';

-- Step 4: Review the employee’s order history for that day to confirm intent or fraud
SELECT PurchaseOrderID, OrderDate, EmployeeID
FROM Purchasing.PurchaseOrderHeader
WHERE EmployeeID IN (
    SELECT EmployeeID 
    FROM Purchasing.PurchaseOrderHeader 
    WHERE OrderDate = '2014-03-25'
)
ORDER BY OrderDate DESC;

Commands completed successfully.

(4 rows affected)

(18 rows affected)

(4 rows affected)

(1484 rows affected)

Total execution time: 00:00:00.068

PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ShipMethodID,OrderDate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,ModifiedDate
2393,4,4,258,1582,5,2014-03-25 00:00:00.000,2014-04-03 00:00:00.000,415.3275,33.2262,10.3832,458.9369,2014-04-03 00:00:00.000
2394,4,4,254,1510,5,2014-03-25 00:00:00.000,2014-04-03 00:00:00.000,143.0415,11.4433,3.576,158.0608,2014-04-03 00:00:00.000
2395,4,4,257,1526,2,2014-03-25 00:00:00.000,2014-04-03 00:00:00.000,28199.325,2255.946,704.9831,31160.2541,2014-04-03 00:00:00.000
2396,4,4,261,1644,1,2014-03-25 00:00:00.000,2014-04-03 00:00:00.000,3713.325,297.066,92.8331,4103.2241,2014-04-03 00:00:00.000


PurchaseOrderID,ProductID,OrderQty,UnitPrice,Quantity
2393,465,3,49.6965,502
2393,465,3,49.6965,569
2393,465,3,49.6965,624
2393,466,3,45.423,504
2393,466,3,45.423,571
2393,466,3,45.423,625
2393,467,3,43.323,505
2393,467,3,43.323,572
2393,467,3,43.323,627
2394,462,3,47.6805,310


PurchaseOrderID,EmployeeID,VendorID,VendorName,CreditRating
2393,258,1582,Inner City Bikes,3
2394,254,1510,International,1
2395,257,1526,International Bicycles,1
2396,261,1644,International Sport Assoc.,1


PurchaseOrderID,OrderDate,EmployeeID
3985,2014-08-03 00:00:00.000,257
3986,2014-08-03 00:00:00.000,261
3993,2014-08-03 00:00:00.000,258
3994,2014-08-03 00:00:00.000,254
3995,2014-08-03 00:00:00.000,257
3996,2014-08-03 00:00:00.000,261
3973,2014-08-02 00:00:00.000,258
3974,2014-08-02 00:00:00.000,254
3975,2014-08-02 00:00:00.000,257
3976,2014-08-02 00:00:00.000,261


### **Schemas Needed to Solve the Case with Table or View Examples**

- Purchasing.PurchaseOrderHeader
    
- Purchasing.PurchaseOrderDetail
    
- Production.ProductInventory
    
- HumanResources.Employee
    
- HumanResources.EmployeeDepartmentHistory
    
- Person.Person
    
- Purchasing.Vendor
    

### **Investigation Notes for Queries and Thoughts to Solve the Case**

- Start by identifying the suspicious purchase order from the PurchaseOrderHeader table.
    
- Use the PurchaseOrderDetail and ProductInventory tables to see if the items were received or recorded.
    
- Join with the Employee and Vendor tables to find out who processed or approved the order.
    
- Check the EmployeeDepartmentHistory to confirm the employee’s role and determine if they had access to purchasing approvals.

## **Case #002: The Vanishing Sales Spike**

## 

**Database**: _AdventureWorksDW2019_

### **Brief Case**

## 

On **June 15, 2013**, the marketing analytics team flagged a major spike in online bike sales. However, the finance department reported no corresponding increase in revenue. This discrepancy has raised concerns of data manipulation or reporting errors. Your task is to investigate the inflated sales entries, verify product legitimacy, and determine whether the activity was a mistake or a deliberate attempt to boost performance metrics.

### **Objectives**

## 

1. Retrieve internet sales from `FactInternetSales` for June 15, 2013, using the `OrderDateKey`.
    
2. Validate that the products sold were legitimate using the `DimProduct` table.
    
3. Identify the employee(s) linked to these transactions using `DimEmployee`.
    
4. Review the employee’s quota and sales territory using `FactSalesQuota` and `DimSalesTerritory` to investigate potential motives.

## **SQL Workspace to Test Queries**

In [4]:
USE AdventureWorksDW2019
GO

-- Step 1: Retrieve internet sales for June 15, 2013
SELECT *
FROM FactInternetSales
WHERE OrderDateKey = 20130615;

-- Step 2: Check product legitimacy
SELECT fis.SalesOrderNumber, fis.ProductKey, p.EnglishProductName, p.ProductLine, p.Status
FROM FactInternetSales AS fis
INNER JOIN DimProduct AS p 
    ON fis.ProductKey = p.ProductKey
WHERE fis.OrderDateKey = 20130615;

-- Step 3: Determine which sales territory had the spike
SELECT DISTINCT dst.SalesTerritoryKey, dst.SalesTerritoryRegion, dst.SalesTerritoryCountry
FROM FactInternetSales AS fis
INNER JOIN DimSalesTerritory AS dst 
    ON fis.SalesTerritoryKey = dst.SalesTerritoryKey
WHERE fis.OrderDateKey = 20130615;

-- Step 4: Get employee(s) assigned to that territory with quotas
SELECT e.EmployeeKey, e.FirstName, e.LastName, e.Title, fq.SalesAmountQuota, fq.CalendarYear, fq.CalendarQuarter, dst.SalesTerritoryRegion
FROM DimEmployee AS e
INNER JOIN FactSalesQuota AS fq 
    ON e.EmployeeKey = fq.EmployeeKey
INNER JOIN DimSalesTerritory AS dst 
    ON e.SalesTerritoryKey = dst.SalesTerritoryKey
WHERE dst.SalesTerritoryKey IN (
    SELECT DISTINCT SalesTerritoryKey
    FROM FactInternetSales
    WHERE OrderDateKey = 20130615
);

Commands completed successfully.

(167 rows affected)

(167 rows affected)

(7 rows affected)

(103 rows affected)

Total execution time: 00:00:00.071

ProductKey,OrderDateKey,DueDateKey,ShipDateKey,CustomerKey,PromotionKey,CurrencyKey,SalesTerritoryKey,SalesOrderNumber,SalesOrderLineNumber,RevisionNumber,OrderQuantity,UnitPrice,ExtendedAmount,UnitPriceDiscountPct,DiscountAmount,ProductStandardCost,TotalProductCost,SalesAmount,TaxAmt,Freight,CarrierTrackingNumber,CustomerPONumber,OrderDate,DueDate,ShipDate
225,20130615,20130627,20130622,12954,1,100,1,SO60179,1,1,1,8.99,8.99,0,0,6.9223,6.9223,8.99,0.7192,0.2248,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
535,20130615,20130627,20130622,12672,1,6,9,SO60180,1,1,1,24.99,24.99,0,0,9.3463,9.3463,24.99,1.9992,0.6248,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
217,20130615,20130627,20130622,12672,1,6,9,SO60180,2,1,1,34.99,34.99,0,0,13.0863,13.0863,34.99,2.7992,0.8748,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
485,20130615,20130627,20130622,23330,1,6,9,SO60181,1,1,1,21.98,21.98,0,0,8.2205,8.2205,21.98,1.7584,0.5495,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
478,20130615,20130627,20130622,23330,1,6,9,SO60181,2,1,1,9.99,9.99,0,0,3.7363,3.7363,9.99,0.7992,0.2498,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
485,20130615,20130627,20130622,22179,1,6,9,SO60182,1,1,1,21.98,21.98,0,0,8.2205,8.2205,21.98,1.7584,0.5495,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
537,20130615,20130627,20130622,18901,1,6,9,SO60183,1,1,1,35.00,35.00,0,0,13.09,13.09,35.00,2.80,0.875,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
538,20130615,20130627,20130622,26030,1,6,9,SO60184,1,1,1,21.49,21.49,0,0,8.0373,8.0373,21.49,1.7192,0.5373,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
530,20130615,20130627,20130622,24243,1,6,9,SO60185,1,1,1,4.99,4.99,0,0,1.8663,1.8663,4.99,0.3992,0.1248,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000
222,20130615,20130627,20130622,24243,1,6,9,SO60185,2,1,1,34.99,34.99,0,0,13.0863,13.0863,34.99,2.7992,0.8748,NULL,NULL,2013-06-15 00:00:00.000,2013-06-27 00:00:00.000,2013-06-22 00:00:00.000


SalesOrderNumber,ProductKey,EnglishProductName,ProductLine,Status
SO60179,225,AWC Logo Cap,S,Current
SO60180,535,LL Mountain Tire,M,Current
SO60180,217,"Sport-100 Helmet, Black",S,Current
SO60181,485,Fender Set - Mountain,M,Current
SO60181,478,Mountain Bottle Cage,M,Current
SO60182,485,Fender Set - Mountain,M,Current
SO60183,537,HL Mountain Tire,M,Current
SO60184,538,LL Road Tire,R,Current
SO60185,530,Touring Tire Tube,T,Current
SO60185,222,"Sport-100 Helmet, Blue",S,Current


SalesTerritoryKey,SalesTerritoryRegion,SalesTerritoryCountry
1,Northwest,United States
4,Southwest,United States
6,Canada,Canada
7,France,France
8,Germany,Germany
9,Australia,Australia
10,United Kingdom,United Kingdom


EmployeeKey,FirstName,LastName,Title,SalesAmountQuota,CalendarYear,CalendarQuarter,SalesTerritoryRegion
282,Linda,Mitchell,Sales Representative,637000.00,2010,4,Southwest
284,Garrett,Vargas,Sales Representative,244000.00,2010,4,Canada
286,Pamela,Ansman-Wolfe,Sales Representative,165000.00,2010,4,Northwest
287,Shu,Ito,Sales Representative,460000.00,2010,4,Southwest
288,José,Saraiva,Sales Representative,525000.00,2010,4,Canada
289,David,Campbell,Sales Representative,226000.00,2010,4,Northwest
282,Linda,Mitchell,Sales Representative,781000.00,2011,1,Southwest
284,Garrett,Vargas,Sales Representative,356000.00,2011,1,Canada
286,Pamela,Ansman-Wolfe,Sales Representative,469000.00,2011,1,Northwest
287,Shu,Ito,Sales Representative,549000.00,2011,1,Southwest


### **Schemas Needed to Solve the Case with Table or View Examples**

- dbo.FactInternetSales
    
- dbo.DimProduct
    
- dbo.DimEmployee
    
- dbo.FactSalesQuota
    
- dbo.DimSalesTerritory
    
- dbo.DimDate _(optional, for date conversions if needed)_
    

### **Investigation Notes for Queries and Thoughts to Solve the Case**

- Start by identifying sales on the specified date using `FactInternetSales` and the `OrderDateKey`.
    
- Use `DimProduct` to verify if the sold products are normally offered online and if they match the product category.
    
- Join with `DimEmployee` to determine who is associated with those sales.
    
- Investigate the employee’s quota history (`FactSalesQuota`) and assigned regions (`DimSalesTerritory`) to detect if performance pressures could have influenced the data spike.

## **Case #003: The Breach in the Books**

**Database**: _WideWorldImporters_

### **Brief Case**

On **January 25, 2016**, the IT security team at WideWorldImporters flagged unauthorized access to sensitive customer order records. The breach appears to have originated internally. A single employee's access pattern triggered multiple alerts when querying data from the sales system outside business hours. You are tasked with investigating the breach, identifying the employee responsible, and determining whether customer data was compromised.

### **Objectives**

1. Retrieve order records from `Sales.Orders` on January 25, 2016.
    
2. Identify the employee responsible for creating or editing the orders using `Application.People`.
    
3. Link each order to customer details via `Sales.Customers` and order lines in `Sales.OrderLines`.
    
4. Review the employee’s access pattern or edit history using the `LastEditedBy` and `LastEditedWhen` fields.

## **SQL Workspace to Test Queries**

In [5]:
USE WideWorldImporters
GO
-- Step 2: Identify employee who created/edited orders using Application.People
SELECT o.OrderID, o.OrderDate, o.LastEditedBy, o.LastEditedWhen, p.FullName, p.PreferredName
FROM Sales.Orders AS o
INNER JOIN Application.People AS p
    ON o.LastEditedBy = p.PersonID
WHERE o.OrderDate = '2016-01-25';

-- Step 3: Link orders to customers and their order lines
SELECT o.OrderID, c.CustomerName, ol.StockItemID, ol.Description, ol.Quantity, ol.UnitPrice
FROM Sales.Orders AS o
INNER JOIN Sales.Customers AS c
    ON o.CustomerID = c.CustomerID
INNER JOIN Sales.OrderLines AS ol
    ON o.OrderID = ol.OrderID
WHERE o.OrderDate = '2016-01-25';

-- Step 4: Review suspicious editing activity (outside business hours)
SELECT o.OrderID, o.OrderDate, o.LastEditedWhen, p.FullName
FROM Sales.Orders AS o
INNER JOIN Application.People AS p
    ON o.LastEditedBy = p.PersonID
WHERE o.OrderDate = '2016-01-25'
  AND (DATEPART(HOUR, o.LastEditedWhen) < 8 OR DATEPART(HOUR, o.LastEditedWhen) > 18);  -- Outside 8AM–6PM


Commands completed successfully.

(88 rows affected)

(246 rows affected)

(0 rows affected)

Total execution time: 00:00:00.123

OrderID,OrderDate,LastEditedBy,LastEditedWhen,FullName,PreferredName
65494,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65495,2016-01-25,2,2016-01-25 12:00:00.0000000,Kayla Woodcock,Kayla
65496,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65497,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65498,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65499,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65500,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65501,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica
65502,2016-01-25,2,2016-01-25 12:00:00.0000000,Kayla Woodcock,Kayla
65503,2016-01-25,9,2016-01-25 11:00:00.0000000,Alica Fatnowna,Alica


OrderID,CustomerName,StockItemID,Description,Quantity,UnitPrice
65539,Alena Kellnerova,27,DBA joke mug - SELECT caffeine FROM mug (Black),7,13.00
65496,"Wingtip Toys (Keosauqua, IA)",17,DBA joke mug - mind if I join you? (Black),7,13.00
65568,"Tailspin Toys (Batson, TX)",53,IT joke mug - keyboard not found … press F1 to continue (Black),7,13.00
65537,"Tailspin Toys (Koontzville, WA)",55,IT joke mug - that behavior is by design (Black),7,13.00
65524,"Tailspin Toys (Tunnelhill, PA)",18,DBA joke mug - daaaaaa-ta (White),1,13.00
65505,"Tailspin Toys (Gasport, NY)",24,DBA joke mug - I will get you in order (White),1,13.00
65540,"Tailspin Toys (Armstrong Creek, WI)",46,Developer joke mug - a foo walks into a bar (White),2,13.00
65569,"Tailspin Toys (Malott, WA)",31,Developer joke mug - Oct 31 = Dec 25 (Black),2,13.00
65524,"Tailspin Toys (Tunnelhill, PA)",39,Developer joke mug - inheritance is the OO way to become wealthy (Black),2,13.00
65526,Ganesh Majumdar,38,Developer joke mug - inheritance is the OO way to become wealthy (White),9,13.00


OrderID,OrderDate,LastEditedWhen,FullName


### **Schemas Needed to Solve the Case with Table or View Examples**

- Sales.Orders
    
- Sales.OrderLines
    
- Sales.Customers
    

### **Investigation Notes for Queries and Thoughts to Solve the Case**

- Start by identifying orders placed on the flagged date from `Sales.Orders`.
    
- Use `LastEditedBy` and `SalespersonPersonID` to link orders to specific users in `Application.People`.
    
- Join with `Sales.Customers` and `Sales.OrderLines` to confirm whether sensitive information was accessed.
    
- Investigate if the same employee repeatedly accessed or edited orders, especially outside normal hours, by checking `LastEditedWhen`.

## **Case #4: The Mug Margin Mystery**

In early January 2013, AdventureWorks' finance team discovered a pattern of **unusually low profit margins** on a specific line of **novelty mugs**. These mugs were sold at prices well below standard markup in several regions, raising suspicion of **internal discounting or price manipulation**. Your task is to analyze sales made on **January 7, 2013**, identify low-profit mug sales, and determine which employee(s) were responsible.

### Objectives

1. Retrieve sales records and filter the results
    
2. Join the data with `Dimension.Employee` using `Salesperson Key` to identify the employees involved.
    
3. Determine the location of the transactions.
    
4. Flag any transactions as suspicious if the `Profit` is below 30 and `Tax Amount` exceeds 5
    

## **SQL Workspace to Test Queries**

In [6]:
USE WideWorldImportersDW;
GO

-- Step 1: Isolate mug sales from Jan 7, 2013
WITH MugSales AS (
    SELECT *
    FROM Fact.Sale
    WHERE [Invoice Date Key] = '2013-01-07'
      AND Description LIKE '%mug%'
      AND Profit < 30
)

-- Step 2: Join with employee and city info, and flag risky transactions
SELECT 
    ms.[Sale Key],
    ms.Description,
    ms.Profit,
    ms.[Tax Amount],
    e.Employee AS Salesperson,
    c.City,
    c.Region,
    CASE 
        WHEN ms.Profit < 30 AND ms.[Tax Amount] > 5 THEN 'Suspicious'
        ELSE 'OK'
    END AS RiskFlag
FROM MugSales ms
JOIN Dimension.Employee e ON ms.[Salesperson Key] = e.[Employee Key]
JOIN Dimension.City c ON ms.[City Key] = c.[City Key]
ORDER BY ms.Profit;


Commands completed successfully.

(22 rows affected)

Total execution time: 00:00:00.637

Sale Key,Description,Profit,Tax Amount,Salesperson,City,Region,RiskFlag
978,DBA joke mug - SELECT caffeine FROM mug (White),8.50,1.95,Taj Shand,Cherryplain,Americas,OK
824,DBA joke mug - I will get you in order (Black),8.50,1.95,Jack Potter,Fieldbrook,Americas,OK
806,DBA joke mug - I will get you in order (White),8.50,1.95,Kayla Woodcock,Sauquoit,Americas,OK
865,Developer joke mug - fun was unexpected at this time (Black),8.50,1.95,Archer Lamble,Inguadona,Americas,OK
977,Developer joke mug - this code was generated by a tool (White),8.50,1.95,Taj Shand,Cherryplain,Americas,OK
972,DBA joke mug - SELECT caffeine FROM mug (Black),8.50,1.95,Jack Potter,Trumansburg,Americas,OK
1012,Developer joke mug - when your hammer is C++ (White),8.50,1.95,Kayla Woodcock,Wounded Knee,Americas,OK
858,Developer joke mug - this code was generated by a tool (White),8.50,1.95,Archer Lamble,Larose,Americas,OK
1022,Developer joke mug - understanding recursion requires understanding recursion (White),8.50,1.95,Archer Lamble,Eagle Valley,Americas,OK
888,Developer joke mug - old C developers never die (Black),8.50,1.95,Amy Trefl,Yaak,Americas,OK


### **Schemas -**

- `Fact.Sale`
    
- `Dimension.Employee`
    
- `Dimension.City`

## **Investigation Notes for Queries**

1. <span style="color: var(--vscode-foreground);">Start by narrowing sales from </span> **January 7, 2013** 
2. <span style="color: var(--vscode-foreground);">Filter the </span> `Description` <span style="color: var(--vscode-foreground);">field to identify mug-related items</span>
3. <span style="color: var(--vscode-foreground);">Flag entries with </span> **low profit** <span style="color: var(--vscode-foreground);">and </span> **high tax** <span style="color: var(--vscode-foreground);">to highlight potential fraud</span>
4. <span style="color: var(--vscode-foreground);">Connect the corresponding low sales to the employees</span>

### **Case #5: The Bubble Wrap Backdoor**

### **Brief Case:**

On **January 7, 2013**, WideWorldImporters recorded an unusually large sale of **Blue Bubble Wrap** — a logistics item typically used internally for shipping and packaging. While sales of such materials are not uncommon, the **quantity** and **profit margin** triggered an alert.  
Even more suspicious, the salesperson who processed the order had never handled packaging materials before. This has raised concerns of **inventory redirection**, where internal-use materials are offloaded through fake customer transactions.  
You’ve been tasked with investigating whether this transaction was legitimate or a sign of internal fraud.

### **Objectives:**

1. Retrieve all sales from **January 7, 2013** where the item description includes `"bubble wrap"`.
    
2. Use a **CTE** to isolate these bubble wrap sales for analysis.
    
3. Join with `Fact.Sale` using **`OUTER APPLY`** to retrieve the **salesperson’s most recent prior transaction**.
    
4. Use a **`CASE`** expression to flag any sale where:
    
- The **previous sale** was not a packaging-related item.
    
- The employee appears to have **suddenly switched** categories (e.g., mugs → logistics)
    

  

## **SQL Workspace to Begin:**

In [7]:
USE WideWorldImportersDW;
GO

-- Step 1: CTE to isolate bubble wrap sales on a known valid date
WITH BubbleWrapSales AS (
    SELECT *
    FROM Fact.Sale
    WHERE [Invoice Date Key] = '2013-01-07'
      AND Description LIKE '%bubble wrap%'
)

-- Step 2: Pull each salesperson's most recent prior transaction using OUTER APPLY
SELECT 
    bws.[Sale Key],
    bws.[Invoice Date Key],
    bws.Description,
    bws.Quantity,
    bws.[Unit Price],
    bws.Profit,
    bws.[Salesperson Key],
    bws.[City Key],
    bws.[Customer Key],
    prev.Description AS PrevDescription,
    prev.[Invoice Date Key] AS PrevInvoiceDate,
    -- Step 3: CASE to flag sales as suspicious
    CASE 
        WHEN prev.Description NOT LIKE '%bubble wrap%' THEN 'Suspicious Switch'
        ELSE 'Normal'
    END AS AnomalyFlag
FROM BubbleWrapSales bws
OUTER APPLY (
    SELECT TOP 1 *
    FROM Fact.Sale s
    WHERE s.[Salesperson Key] = bws.[Salesperson Key]
      AND s.[Invoice Date Key] < bws.[Invoice Date Key]
    ORDER BY s.[Invoice Date Key] DESC
) AS prev
ORDER BY bws.Quantity DESC;


Commands completed successfully.

(27 rows affected)

Total execution time: 00:00:00.060

Sale Key,Invoice Date Key,Description,Quantity,Unit Price,Profit,Salesperson Key,City Key,Customer Key,PrevDescription,PrevInvoiceDate,AnomalyFlag
1034,2013-01-07,10 mm Anti static bubble wrap (Blue) 20m,100,42.00,1900.00,19,59352,0,Developer joke mug - when your hammer is C++ (Black),2013-01-05,Suspicious Switch
924,2013-01-07,32 mm Double sided bubble wrap 20m,100,37.00,2300.00,7,63191,282,IT joke mug - keyboard not found … press F1 to continue (Black),2013-01-04,Suspicious Switch
778,2013-01-07,20 mm Double sided bubble wrap 50m,100,108.00,9200.00,7,61111,38,IT joke mug - keyboard not found … press F1 to continue (Black),2013-01-04,Suspicious Switch
1081,2013-01-07,32 mm Anti static bubble wrap (Blue) 10m,80,32.00,1280.00,11,56844,0,Developer joke mug - fun was unexpected at this time (White),2013-01-05,Suspicious Switch
901,2013-01-07,10 mm Double sided bubble wrap 50m,80,105.00,3920.00,15,75266,236,IT joke mug - keyboard not found … press F1 to continue (Black),2013-01-05,Suspicious Switch
969,2013-01-07,20 mm Anti static bubble wrap (Blue) 50m,80,102.00,3760.00,6,70600,368,Developer joke mug - understanding recursion requires understanding recursion (White),2013-01-05,Suspicious Switch
984,2013-01-07,20 mm Double sided bubble wrap 50m,80,108.00,7360.00,23,38005,381,"Developer joke mug - (hip, hip, array) (Black)",2013-01-05,Suspicious Switch
773,2013-01-07,32 mm Double sided bubble wrap 50m,70,112.00,3710.00,7,45502,29,IT joke mug - keyboard not found … press F1 to continue (Black),2013-01-04,Suspicious Switch
812,2013-01-07,32 mm Anti static bubble wrap (Blue) 10m,70,32.00,1120.00,12,74557,94,Developer joke mug - a foo walks into a bar (White),2013-01-05,Suspicious Switch
944,2013-01-07,20 mm Double sided bubble wrap 10m,70,18.00,700.00,8,51573,336,DBA joke mug - SELECT caffeine FROM mug (Black),2013-01-05,Suspicious Switch


### **Schemas -**

- `Fact.Sale` 
    
- `Dimension.Employee` 
    
- `Dimension.Customer`
    

**Investigation Notes:**

1. The use of `OUTER APPLY` helps analyze sales contextually     
2. The `CASE` logic simulates a **red-flag detection system**.
3. Consider building this logic into a **View** or **Stored Procedure** if your team wants to run fraud scans regularly.

## **Case #6: The Cap Trap**

### **Brief Case:**

Internal audits at AdventureWorks have flagged multiple customers placing **suspiciously frequent orders** of the low-cost product: **AWC Logo Cap**. On the surface, the item appears harmless — but deeper analysis shows that some customers have ordered it **12+ times**, with **over 100 units** each. <span style="color: var(--vscode-foreground);">You’ve been asked to develop a system to </span> **detect and flag** <span style="color: var(--vscode-foreground);">such repeat customer-product patterns using AdventureWorks2019 data.</span>

### **Objectives:**

1. Create a **View** that isolates all `"AWC Logo Cap"` sales across all customers.
    
2. Build a **Table-Valued Function** that returns customers who ordered the cap more than a given number of times or units.
    
3. Wrap it all in a **Stored Procedure** that outputs:
    
    - Customer ID
        
    - Total Order Count
        
    - Total Quantity Ordered
        
    - Flag (`'Suspicious'` or `'Normal'`) using a `CASE` expression
        
4. Use this procedure to scan for **repeat cap abuse** using a threshold of **\\\>10 orders** or **\\\>100 units**.
    

## **SQL Workspace to Begin:**

In [8]:
USE AdventureWorks2019;
GO

WITH CapSales AS (
    SELECT 
        h.CustomerID,
        d.OrderQty,
        p.Name AS ProductName
    FROM Sales.SalesOrderHeader h
    JOIN Sales.SalesOrderDetail d ON h.SalesOrderID = d.SalesOrderID
    JOIN Production.Product p ON d.ProductID = p.ProductID
    WHERE p.Name = 'AWC Logo Cap'
),
AbuseSummary AS (
    SELECT 
        CustomerID,
        COUNT(*) AS OrderCount,
        SUM(OrderQty) AS TotalQuantity
    FROM CapSales
    GROUP BY CustomerID
    HAVING COUNT(*) > 10 OR SUM(OrderQty) > 100
)
SELECT 
    CustomerID,
    OrderCount,
    TotalQuantity,
    CASE 
        WHEN OrderCount > 10 OR TotalQuantity > 100 THEN 'Suspicious'
        ELSE 'Normal'
    END AS FraudFlag
FROM AbuseSummary
ORDER BY TotalQuantity DESC;


Commands completed successfully.

(12 rows affected)

Total execution time: 00:00:00.067

CustomerID,OrderCount,TotalQuantity,FraudFlag
29705,12,113,Suspicious
29722,12,108,Suspicious
29992,12,96,Suspicious
29637,12,86,Suspicious
29950,11,81,Suspicious
30117,12,66,Suspicious
29523,12,55,Suspicious
29966,11,52,Suspicious
29825,12,47,Suspicious
29522,12,44,Suspicious


## **Schemas -**

- Sales.SalesOrderHeader
- Sales.SalesOrderDetail
- Production.Product

## **Investigation Notes -**

- Start by isolating all sales of the "AWC Logo Cap" product using a join table
- Use a Common Table Expression (CTE) to focus only on these filtered transactions.
- Group the sales by `CustomerID` to calculate the number of times each customer ordered the cap 
- Add another CTE to select only customers who exceed predefined thresholds (more than 10 orders or more than 100 units).
- Use a `CASE` expression to flag these customers as either `'Suspicious'` or `'Normal'` depending on whether their order activity crosses the threshold.

## **Case #7: Tax and Freight Fraud**

AdventureWorks has observed anomalies in several sales orders, where the **TaxAmt** and **Freight** do not align with what was expected based on the subtotal of the orders. Some orders have a **surprisingly low tax amount**, while others have **inexplicably low freight** for orders with a large total.

Management has flagged these discrepancies for investigation, suspecting potential **fraudulent activity** or **data errors** that could affect profits and customer billing.

### **Objectives:**

1. **Calculate Expected Tax** for each order, assuming an 8% standard tax rate.
    
2. **Identify Orders** with tax amounts that are either too low or too high compared to the expected amount.
    
3. **Flag Orders** where **Freight** is abnormally low for a given `SubTotal` (i.e., freight under $5 for orders over $100).
    
4. Identify orders where the **TotalDue** doesn’t match the expected sum (Subtotal + TaxAmt + Freight).
    
5. **Return** a list of suspicious orders, flagging them as **“Suspicious Tax”**, **“Suspicious Freight”**, or **“Discrepancy”** for TotalDue mismatches.
    

## **SQL Workspace to Start:**

In [3]:
USE AdventureWorks2019;
GO

-- Step 1: CTE to calculate expected tax and identify discrepancies
WITH TaxAndFreightCheck AS (
    SELECT 
        SalesOrderID,
        OrderDate,
        SubTotal,
        TaxAmt,
        Freight,
        TotalDue,
        CustomerID,
        CreditCardID,
        -- Assuming an 8% standard tax rate for comparison
        SubTotal * 0.08 AS ExpectedTax,
        CASE 
            WHEN TaxAmt < SubTotal * 0.08 * 0.5 THEN 'Suspicious Tax'  -- Flag tax discrepancies below 50% expected
            WHEN TaxAmt > SubTotal * 0.08 * 1.5 THEN 'Suspicious Tax'  -- Flag tax discrepancies above 150% expected
            ELSE 'Normal'
        END AS TaxStatus,
        CASE 
            WHEN Freight < 5 AND SubTotal > 100 THEN 'Suspicious Freight' -- Low freight for high subtotal
            ELSE 'Normal'
        END AS FreightStatus,
        -- Calculate total due (tax + subtotal + freight)
        TotalDue - (SubTotal + TaxAmt + Freight) AS Difference
    FROM Sales.SalesOrderHeader
)

-- Step 2: Retrieve the final filtered data
SELECT 
    SalesOrderID,
    OrderDate,
    SubTotal,
    TaxAmt,
    Freight,
    TotalDue,
    CustomerID,
    CreditCardID,
    ExpectedTax,
    TaxStatus,
    FreightStatus,
    Difference
FROM TaxAndFreightCheck
WHERE TaxStatus = 'Suspicious Tax' 
   OR FreightStatus = 'Suspicious Freight'
   OR Difference > 1  -- Flag if the difference between expected and actual TotalDue is greater than 1
ORDER BY Difference DESC;


Commands completed successfully.

(1221 rows affected)

Total execution time: 00:00:00.165

SalesOrderID,OrderDate,SubTotal,TaxAmt,Freight,TotalDue,CustomerID,CreditCardID,ExpectedTax,TaxStatus,FreightStatus,Difference
44785,2011-10-31 00:00:00.000,144.202,12.4975,3.9055,160.605,29674,17073,11.536160,Normal,Suspicious Freight,0.00
46323,2012-04-30 00:00:00.000,2238.4935,511.6525,159.8914,2910.0374,29632,13278,179.079480,Suspicious Tax,Normal,0.00
46327,2012-04-30 00:00:00.000,13125.6019,2309.0728,721.5852,16156.2599,29622,7829,1050.048152,Suspicious Tax,Normal,0.00
46330,2012-04-30 00:00:00.000,13129.4681,2307.2596,721.0186,16157.7463,29950,5110,1050.357448,Suspicious Tax,Normal,0.00
46332,2012-04-30 00:00:00.000,5593.2771,1277.1232,399.101,7269.5013,29803,8552,447.462168,Suspicious Tax,Normal,0.00
46334,2012-04-30 00:00:00.000,40708.4413,7996.6623,2498.957,51204.0606,29722,12698,3256.675304,Suspicious Tax,Normal,0.00
46342,2012-04-30 00:00:00.000,14076.1511,2839.1443,887.2326,17802.528,29748,11555,1126.092088,Suspicious Tax,Normal,0.00
46345,2012-04-30 00:00:00.000,17671.8447,3268.5018,1021.4068,21961.7533,29504,18100,1413.747576,Suspicious Tax,Normal,0.00
46346,2012-04-30 00:00:00.000,8912.6663,2039.5265,637.352,11589.5448,29655,14064,713.013304,Suspicious Tax,Normal,0.00
46348,2012-04-30 00:00:00.000,24607.478,4371.1602,1365.9875,30344.6257,29690,7392,1968.598240,Suspicious Tax,Normal,0.00


### **Schemas -**

### SalesOrderHeader

SalesOrderDetail  
ProductCategory

PurchaseOrderDetail

### **Investigation Notes:**

- **CTE** is used to calculate the expected tax based on the `SubTotal` and compare it against the `TaxAmt`.
    
- The **`CASE`** expression flags transactions with **suspect tax amounts** and **low freight costs** for large orders.
    
- **Freight anomalies** are flagged for orders where the freight is less than $5 while the order value exceeds $100.
    
- The **TotalDue** discrepancy is checked, comparing the difference between what is expected (Subtotal + Tax + Freight) and what was charged.
    
- This query returns **all suspicious activity** based on the defined thresholds, with detailed flags like `'Suspicious Tax'` and `'Suspicious Freight'`.